# Embedding Quality Evaluation Pipeline

This notebook demonstrates how to use the `nlpie` library to preprocess embeddings, evaluate their quality across multiple dimensions, and compare different models side-by-side.

In [1]:
import math
import sys
import numpy as np

# Ensure nlpie is in path if running from the repo root
sys.path.insert(0, "..")
sys.path.insert(0, "../../python")

from nlpie import EmbeddingPreprocessor
from metrics.quality import evaluate_embedding_quality, compare_models

## 1. Synthetic Data Generation

First, we'll create two synthetic embedding models. 
- **Model A (Baseline)**: Noisy clusters.
- **Model B (Finetuned)**: Tightly grouped clusters with clear separation.

In [2]:
np.random.seed(42)
n_samples = 300
n_dims = 64
n_classes = 5

# Ground truth labels
labels = np.random.randint(0, n_classes, n_samples)

# Generate class centers
centers_a = np.random.randn(n_classes, n_dims)
centers_b = np.random.randn(n_classes, n_dims) * 2.0  # More spread out

# Generate embeddings
emb_a = centers_a[labels] + np.random.randn(n_samples, n_dims) * 1.5  # Noisy
emb_b = centers_b[labels] + np.random.randn(n_samples, n_dims) * 0.5  # Clean

## 2. Preprocessing

Before evaluating the embeddings, it's often beneficial to normalize them. We'll use the `EmbeddingPreprocessor` to mean-center and L2-normalize the embeddings. Because `nlpie` is backed by Rust, this is extremely fast.

In [3]:
preprocessor = EmbeddingPreprocessor()

# Normalize Model A
emb_a_centered, _ = preprocessor.mean_center(emb_a)
emb_a_norm = preprocessor.l2_normalize_rows(emb_a_centered)

# Normalize Model B
emb_b_centered, _ = preprocessor.mean_center(emb_b)
emb_b_norm = preprocessor.l2_normalize_rows(emb_b_centered)

## 3. Synthetic Projections & Retrieval Queries

To evaluate projection quality and retrieval, we need a low-dimensional projection (e.g. from UMAP/PCA) and some retrieval queries.

In [4]:
# Simulate 2D projections (in reality, use umap-learn or sklearn.decomposition.PCA)
# For Model A, we'll just add some noise to the first 2 dimensions.
low_dim_a = np.array(emb_a_norm)[:, :2] + np.random.randn(n_samples, 2) * 0.1
low_dim_b = np.array(emb_b_norm)[:, :2]

# Simulate retrieval queries.
# Let's say we have 10 queries, and for each query we retrieved the top 5 documents.
n_queries = 10
retrieved_a = [np.random.choice(n_samples, 5, replace=False).tolist() for _ in range(n_queries)]
retrieved_b = [np.random.choice(n_samples, 5, replace=False).tolist() for _ in range(n_queries)]

# Ground-truth relevant documents for each query (2 relevant docs per query)
relevant = [np.random.choice(n_samples, 2, replace=False).tolist() for _ in range(n_queries)]

## 4. Single Model Evaluation

Let's evaluate the "Finetuned" model (Model B) across all metric categories simultaneously.

In [5]:
report_b = evaluate_embedding_quality(
    embeddings=emb_b_norm,
    labels=labels.tolist(),                 # Clustering metrics
    low_dim=low_dim_b.tolist(),             # Projection metrics
    projection_k_values=[5, 10],
    retrieved=retrieved_b,                  # Retrieval metrics
    relevant=relevant,
    retrieval_k_values=[1, 3, 5],
    hubness_k=5,                            # Geometry metrics
    model_name="Model B (Finetuned)"
)

print(report_b)

Embedding Quality Report: Model B (Finetuned)
  Samples=300  Dims=64
Intrinsic (Cosine Similarity)
  mean=-0.0023  std=0.4901  min=-0.5281  max=0.9744
Clustering
  ARI=1.0000  NMI=1.0000  Purity=1.0000
  Silhouette=0.7619  Calinski-Harabasz=1170.25
Geometry & Pathology
  Effective Rank=5.52  Mean Sim-to-Mean=0.0317
  Hubness Skewness=1.8137 (k=5)
Projection Quality
  k=  5  Trustworthiness=0.8969  Continuity=0.9097
  k= 10  Trustworthiness=0.9078  Continuity=0.9194
Retrieval Quality
  k=  1  R@K=0.0000  P@K=0.0000  MRR=0.0250  nDCG=0.0000  Cov=0.0000
  k=  3  R@K=0.0000  P@K=0.0000  MRR=0.0250  nDCG=0.0000  Cov=0.0000
  k=  5  R@K=0.0500  P@K=0.0200  MRR=0.0250  nDCG=0.0264  Cov=0.0500


## 5. Multi-Model Comparison

Often, we want to compare a baseline model with a new finetuned model. `compare_models` makes this easy.

In [6]:
comparison_reports = compare_models(
    models={
        "Model A (Baseline)": emb_a_norm,
        "Model B (Finetuned)": emb_b_norm
    },
    labels=labels.tolist(),
    low_dims={
        "Model A (Baseline)": low_dim_a.tolist(),
        "Model B (Finetuned)": low_dim_b.tolist(),
    },
    projection_k_values=[10],
    retrieved={
        "Model A (Baseline)": retrieved_a,
        "Model B (Finetuned)": retrieved_b,
    },
    relevant=relevant,
    retrieval_k_values=[5],
)

for report in comparison_reports:
    print(report)
    print("\n" * 2)

Embedding Quality Report: Model A (Baseline)
  Samples=300  Dims=64
Intrinsic (Cosine Similarity)
  mean=-0.0033  std=0.1849  min=-0.5339  max=0.6665
Clustering
  ARI=1.0000  NMI=1.0000  Purity=1.0000
  Silhouette=0.1559  Calinski-Harabasz=30.73
Geometry & Pathology
  Effective Rank=42.66  Mean Sim-to-Mean=0.0080
  Hubness Skewness=1.3089 (k=5)
Projection Quality
  k= 10  Trustworthiness=0.5723  Continuity=0.5908
Retrieval Quality
  k=  5  R@K=0.0000  P@K=0.0000  MRR=0.0000  nDCG=0.0000  Cov=0.0000



Embedding Quality Report: Model B (Finetuned)
  Samples=300  Dims=64
Intrinsic (Cosine Similarity)
  mean=-0.0023  std=0.4901  min=-0.5281  max=0.9744
Clustering
  ARI=1.0000  NMI=1.0000  Purity=1.0000
  Silhouette=0.7619  Calinski-Harabasz=1170.25
Geometry & Pathology
  Effective Rank=5.52  Mean Sim-to-Mean=0.0317
  Hubness Skewness=1.8137 (k=5)
Projection Quality
  k= 10  Trustworthiness=0.9078  Continuity=0.9194
Retrieval Quality
  k=  5  R@K=0.0500  P@K=0.0200  MRR=0.0250  nDCG=0.0264